
CodeReview-Bench: An automated benchmark synthesis engine using open-source LLMs to generate structured, ground-truth evaluation datasets for automated code review and deployment gatekeeping.



In [ ]:
!pip install -q bitsandbytes accelerate transformers==4.57.6

In [ ]:
# --- Imports ---

from IPython.display import  display, update_display
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, TextIteratorStreamer
import torch
import threading
from google.colab import userdata

In [ ]:
# --- Constants ---

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
# --- Authentication ---
# assuming you've already added HF_TOKEN in Google Colab's Secret


HUGGINGFACE_TOKEN = userdata.get('HF_TOKEN')
login(token=HUGGINGFACE_TOKEN)


In [ ]:
# --- Model Loading Logic ---

def load_model_and_tokenizer(model_id: str, auth_token: str = None):
    """Loads tokenizer and causal LM directly to CUDA with 4-bit NF4 quantization."""
    # 1. Configure 4-bit quantization
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    # 2. Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 3. Load model directly onto CUDA with quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map="auto", token=auth_token
    )

    return tokenizer, model


In [ ]:
tokenizer, model = load_model_and_tokenizer(QWEN_MODEL, HUGGINGFACE_TOKEN)

In [ ]:
system_prompt = """You are a synthetic dataset generator.
Your job is to invent realistic Python code snippets and label them for code review training.
Output strictly a JSON list containing 4 examples with:
- "code": a realistic 3-6 line Python snippet
- "flaw_type": "hardcoded_secret" | "sql_injection" | "bad_error_handling" | "none"
- "verdict": "Block Deployment" | "Request Changes" | "Merge"
- "reason": brief explanation
"""

In [ ]:
def chat(message, history):
    # 1. Format history into Hugging Face chat template format
    messages = [{"role": "system", "content": system_prompt}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    # 2. Tokenize with attention mask
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # 3. Generate
    input_length = inputs["input_ids"].shape[1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # 4. Slice out only the new tokens
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response

In [ ]:
import threading
from transformers import TextIteratorStreamer

system_prompt = """You are a helpful assistant for pharmacists in a community pharmacy.
You review prescriptions, recommend whether to dispense or not, and call out drug-drug interactions."""

def stream_response(message, history):
    # 1. Guard against None or empty history
    history = history or []

    # 2. Build standard chat messages format
    clean_messages = [{"role": "system", "content": system_prompt}]

    for turn in history:
        # Handles both older Gradio [user_msg, bot_msg] and newer dict formats
        if isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_text, bot_text = turn
            if user_text:
                clean_messages.append({"role": "user", "content": str(user_text)})
            if bot_text:
                clean_messages.append({"role": "assistant", "content": str(bot_text)})
        elif isinstance(turn, dict) and "role" in turn and "content" in turn:
            clean_messages.append({"role": turn["role"], "content": str(turn["content"] or "")})

    # Add the current user prompt
    clean_messages.append({"role": "user", "content": str(message or "")})

    # 3. Tokenize inputs to the model device
    inputs = tokenizer.apply_chat_template(
        clean_messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # 4. Set up streamer
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    # 5. EOS & Pad IDs
    eos_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is not None and im_end_id != tokenizer.unk_token_id and im_end_id not in eos_ids:
        eos_ids.append(im_end_id)

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        eos_token_id=eos_ids,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

    # 6. Run generation in thread
    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # 7. Yield accumulated text for Gradio streaming
    partial_text = ""
    for chunk in streamer:
        cleaned_chunk = chunk.replace("<|im_end|>", "").replace("<|endoftext|>", "")
        partial_text += cleaned_chunk
        yield partial_text

    thread.join()

In [ ]:
!pip install -q --upgrade gradio

In [11]:
import gradio as gr


demo = gr.ChatInterface(fn=stream_response)

gr.close_all()
demo.launch(debug=True)

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a8ffc7f887dfff6e53.gradio.live
